In [2]:
import numpy as np
import pandas as pd

raw_path =r'C:\Users\sahil\projects\boston-mbta-ridership\data\raw\ACS Data\acs_2024_suffolk_tracts.csv'
clean_path =r'C:\Users\sahil\projects\boston-mbta-ridership\data\processed\acs_2024_tracts_clean.csv'

acs = pd.read_csv(raw_path, dtype={"state": str, "county": str, "tract": str})


In [3]:
rename_map = {
    # ---- Tract name ----
    'NAME':        'tract_name',             # e.g. "Census Tract 1.01; Suffolk County; Massachusetts"

    # ---- Income and poverty ----
    'B19013_001E': 'median_hh_income',       # median household income, in dollars
    'B17001_001E': 'pov_universe',           # people whose poverty status could be determined (denominator)
    'B17001_002E': 'pov_below',              # people living below the federal poverty line

    # ---- Vehicles ----
    'B08201_001E': 'households',             # total households (denominator)
    'B08201_002E': 'hh_no_vehicle',          # households with no car, truck or van available

    # ---- Race and ethnicity ----
    'B03002_001E': 'pop_total',              # total population (denominator)
    'B03002_003E': 'pop_nh_white',           # non-Hispanic White
    'B03002_004E': 'pop_nh_black',           # non-Hispanic Black or African American
    'B03002_006E': 'pop_nh_asian',           # non-Hispanic Asian
    'B03002_012E': 'pop_hispanic',           # Hispanic or Latino, any race

    # ---- Commuting mode ----
    'B08301_001E': 'workers',                # workers 16 and older (denominator)
    'B08301_010E': 'workers_transit',        # workers who commute by public transit (not taxi)

    # ---- Commute time ----
    'B08303_001E': 'commuters',              # workers who travel to work, excludes work-from-home (denominator)
    'B08303_011E': 'commute_45_59min',       # commute of 45 to 59 minutes
    'B08303_012E': 'commute_60_89min',       # commute of 60 to 89 minutes
    'B08303_013E': 'commute_90plus_min',     # commute of 90 minutes or more

    # ---- Housing ----
    'B25003_001E': 'occupied_units',         # occupied housing units (denominator)
    'B25003_003E': 'renter_units',           # renter-occupied housing units
    'B25070_001E': 'renter_hh',              # renter households (denominator for rent burden)
    'B25070_007E': 'rent_30_35pct',          # rent is 30.0-34.9% of household income
    'B25070_008E': 'rent_35_40pct',          # rent is 35.0-39.9% of household income
    'B25070_009E': 'rent_40_50pct',          # rent is 40.0-49.9% of household income
    'B25070_010E': 'rent_50plus_pct',        # rent is 50% or more of household income

    # ---- Language ----
    'C16002_001E': 'lang_households',        # total households (denominator for language)
    'C16002_004E': 'limited_eng_spanish',    # limited-English households, Spanish-speaking
    'C16002_007E': 'limited_eng_indo_euro',  # limited-English households, other Indo-European languages
    'C16002_010E': 'limited_eng_asian_pac',  # limited-English households, Asian and Pacific Island languages
    'C16002_013E': 'limited_eng_other',      # limited-English households, all other languages

    # ---- Employment ----
    'B23025_003E': 'labor_force',            # civilian labor force, 16 and older (denominator)
    'B23025_005E': 'unemployed',             # unemployed people in the labor force

    # ---- Geography codes ----
    'state':       'state_fips',             # state code: 25 = Massachusetts
    'county':      'county_fips',            # county code: 025 = Suffolk County
    'tract':       'tract_code',             # 6-digit tract code, e.g. 000101 = tract 1.01
}

acs = acs.rename(columns=rename_map)

In [4]:
# GEOID = state (2 digits) + county (3) + tract (6) = 11 digits, e.g. "25025000101"
# This is the key that links ACS data to the TIGER tract map
acs['GEOID'] = acs['state_fips'] + acs['county_fips'] + acs['tract_code']


drop_cols = [
    'lang_households', 'limited_eng_spanish', 'limited_eng_indo_euro',
    'limited_eng_asian_pac', 'limited_eng_other',
    'state_fips', 'county_fips',
]

acs = acs.drop(columns=drop_cols)

print(acs.shape)

acs.columns.tolist()

(235, 28)


['tract_name',
 'median_hh_income',
 'pov_universe',
 'pov_below',
 'households',
 'hh_no_vehicle',
 'pop_total',
 'pop_nh_white',
 'pop_nh_black',
 'pop_nh_asian',
 'pop_hispanic',
 'workers',
 'workers_transit',
 'commuters',
 'commute_45_59min',
 'commute_60_89min',
 'commute_90plus_min',
 'occupied_units',
 'renter_units',
 'renter_hh',
 'rent_30_35pct',
 'rent_35_40pct',
 'rent_40_50pct',
 'rent_50plus_pct',
 'labor_force',
 'unemployed',
 'tract_code',
 'GEOID']

In [5]:
# Keep only Boston: drop Chelsea (16xx), Revere (17xx), Winthrop (18xx), and non-residential tracts (98xx, 99xx)
drop_prefixes = ['16', '17', '18', '98', '99']
acs_boston = acs[acs['tract_code'].str[:2].isin(drop_prefixes) == False].reset_index(drop=True)
print(acs_boston.shape)   # expect (192, 28)

(192, 28)


In [6]:
# Replace the Census missing-value code with a real missing value (NaN)
acs_boston['median_hh_income'] = acs_boston['median_hh_income'].replace(-666666666, np.nan)

# Drop the tract with no residents
acs_boston = acs_boston[acs_boston['pop_total'] > 0].reset_index(drop=True)
print(acs_boston.shape)   # expect (191, 28)

(191, 28)


In [7]:
# Turn counts into shares (0 to 1). Tracts with a 0 denominator become NaN.
a = acs_boston
a['pct_poverty']        = a['pov_below'] / a['pov_universe']
a['pct_no_vehicle']     = a['hh_no_vehicle'] / a['households']
a['pct_minority']       = 1 - a['pop_nh_white'] / a['pop_total']     # everyone except non-Hispanic White
a['pct_black']          = a['pop_nh_black'] / a['pop_total']
a['pct_hispanic']       = a['pop_hispanic'] / a['pop_total']
a['pct_transit']        = a['workers_transit'] / a['workers']
a['pct_long_commute']   = (a['commute_45_59min'] + a['commute_60_89min'] + a['commute_90plus_min']) / a['commuters']
a['pct_renters']        = a['renter_units'] / a['occupied_units']
a['pct_rent_burdened']  = (a['rent_30_35pct'] + a['rent_35_40pct'] + a['rent_40_50pct'] + a['rent_50plus_pct']) / a['renter_hh']
a['unemployment_rate']  = a['unemployed'] / a['labor_force']

# Check: averages should be sensible (all between 0 and 1)
a.filter(like='pct_').describe().round(2)

,pct_poverty,pct_no_vehicle,pct_minority,pct_black,pct_hispanic,pct_transit,pct_long_commute,pct_renters,pct_rent_burdened
count,191.00,190.00,191.00,191.00,191.00,191.00,191.00,190.00,190.00
mean,0.18,0.35,0.53,0.17,0.18,0.23,0.21,0.66,0.47
std,0.12,0.19,0.27,0.19,0.15,0.11,0.10,0.21,0.13
min,0.01,0.02,0.03,0.00,0.01,0.03,0.01,0.13,0.18
25%,0.08,0.19,0.31,0.03,0.08,0.15,0.12,0.52,0.39
50%,0.15,0.33,0.48,0.08,0.14,0.21,0.20,0.66,0.49
75%,0.26,0.48,0.76,0.26,0.24,0.30,0.28,0.83,0.55
max,0.59,0.81,0.99,0.74,0.78,0.59,0.56,1.00,0.84


In [12]:
acs_boston.to_csv(clean_path, index=False)
print('Saved to', clean_path)

Saved to C:\Users\sahil\projects\boston-mbta-ridership\data\processed\acs_2024_tracts_clean.csv
